In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

## Swain vs Alabama

In [ ]:
population_proportions = make_array(.26, .74)

def panel_proportion():
    return sample_proportions(100, population_proportions).item(0)

panels = make_array()

num_simulations = 10000

for i in np.arange(num_simulations):
    panels = np.append(panels, panel_proportion() * 100)
    
Table().with_column('Number of Black Men on Panel of 100', panels).hist(bins=np.arange(5.5,40.))
plots.axvline(8, color='red', lw=2);

In [ ]:
np.mean(panels <= 8)

## Decisions and Uncertainty

In [ ]:
bike = Table.read_table('/home/jovyan/shared/Data/NCDOT_BikePedCrash.csv').select('BikeAge', 'County')
bike

In [ ]:
np.mean(bike.column('BikeAge'))

In [ ]:
bike.group('BikeAge').sort('BikeAge', descending = True).show()

In [ ]:
type(bike.column('BikeAge').item(0))

In [ ]:
bike_with_ages = bike.where('BikeAge', are.not_contained_in(make_array('999', 'Unknown', '70+')))
bike_with_ages = bike_with_ages.with_column('BikeAge', bike_with_ages.apply(int, 'BikeAge'))
bike_with_ages

In [ ]:
bike_age_mean = np.mean(bike_with_ages.column('BikeAge'))
bike_age_mean

In [ ]:
bike_with_ages.group('County').where('County', 'Orange')

In [ ]:
observed_size = bike_with_ages.group('County').where('County', 'Orange').column('count').item(0)
observed_size

In [ ]:
bike_with_ages.group('County', np.average).where('County', 'Orange')

In [ ]:
observed_average = bike_with_ages.group('County', np.average).where('County', 'Orange').column('BikeAge average').item(0)
observed_average

In [ ]:
observed_average_diff = abs(bike_age_mean - observed_average)
observed_average_diff

In [ ]:
random_sample = bike_with_ages.sample(observed_size, with_replacement=False)
random_sample

In [ ]:
abs(bike_age_mean - np.average(random_sample.column('BikeAge')))

In [ ]:
def random_sample_bikeage_avg_diff():
    """..."""
    random_sample = bike_with_ages.sample(observed_size, with_replacement=False)
    return abs(bike_age_mean - np.average(random_sample.column('BikeAge')))

In [ ]:
random_sample_bikeage_avg_diff()

In [ ]:
# Simulate 10,000 copies of the test statistic

sample_diff_averages = make_array()

for i in np.arange(10000):
    sample_diff_averages = np.append(sample_diff_averages, random_sample_bikeage_avg_diff())    

In [ ]:
# Compare the simulated distribution of the statistic
# and the actual observed statistic

averages_tbl = Table().with_column('Random Sample Average Differences', sample_diff_averages)
averages_tbl.hist(bins = 20)
plots.axvline(observed_average_diff, color='red', lw=2);

In [ ]:
np.mean(sample_diff_averages>= observed_average_diff)

In [ ]:
# 5% of 10,000 = 500

five_percent_point = averages_tbl.sort(0).column(0).item(9500)
five_percent_point

In [ ]:
averages_tbl.hist(bins = 20)
plots.axvline(observed_average_diff, color='red', lw=2);
plots.axvline(five_percent_point, color='gold', lw=2);
plots.title('Area to the left of the gold line: 5%');